# Sprint 7 - GAT/GATv2 Attention Runner

**Runner-only notebook.** Model, loss, sampler, training, evaluation, plotting, and reporting logic stays in the repository under `src/`, `scripts/`, and `configs/`.

Execution plan: `docs/exec-plans/active/007-sprint7-gat-gatv2-attention.md`  
Runner boundary: `colab/README.md`

Before starting, confirm that the approved Sprint 7 code revision is pushed and that Drive contains the Sprint 5 Graph A artifacts with the `S5F2_energy` edge table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/YasinEkici/crispr-gnn-offtarget.git"
REPO_DIR="/content/crispr-gnn-offtarget"
GIT_REF="sprint7/gat-gatv2"
if [ -d "$REPO_DIR/.git" ]; then
  cd "$REPO_DIR"
  git fetch origin "$GIT_REF"
  git checkout "$GIT_REF"
  git pull --ff-only origin "$GIT_REF"
else
  git clone --branch "$GIT_REF" "$REPO_URL" "$REPO_DIR"
  cd "$REPO_DIR"
fi
git rev-parse --short HEAD


In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
python -m pip install -q uv
uv sync
uv run python - <<'PY'
import torch
import torch_geometric
print('torch', torch.__version__)
print('torch_geometric', torch_geometric.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
PY


In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
ALT_DRIVE_ROOT="/content/drive/MyDrive/crispr-gnn-offtarget"
if [ ! -d "$DRIVE_ROOT" ] && [ -d "$ALT_DRIVE_ROOT" ]; then
  DRIVE_ROOT="$ALT_DRIVE_ROOT"
fi
GRAPH_SOURCE="$DRIVE_ROOT/data/processed/graphs/sprint5"
test -d "$GRAPH_SOURCE/graph_a_minimal_physical_target"
mkdir -p data/processed/graphs/sprint5
rsync -a "$GRAPH_SOURCE/" data/processed/graphs/sprint5/
find data/processed/graphs/sprint5/graph_a_minimal_physical_target -maxdepth 2 -type f | sort | head -30


In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
PYTHONPATH=src uv run python - <<'PY'
from pathlib import Path
from crispr_gnn.graph.graph_schemas import GRAPH_A
from crispr_gnn.graph.pyg_dataset import Sprint3HeteroDataLoader
materialized = Sprint3HeteroDataLoader(Path('data/processed/graphs/sprint5')).load(GRAPH_A)
manifest = materialized.manifest
print('graph_name', manifest.get('graph_name'))
print('split_id', manifest.get('split_id'))
print('label_scheme', manifest.get('label_scheme'))
print('feature_tables', manifest.get('feature_tables'))
feature_tables = manifest.get('feature_tables', {})
if 'S5F2_energy' not in feature_tables and 's5f2_energy' not in feature_tables:
    raise SystemExit('Missing S5F2_energy feature table')
PY


In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_ID="sprint7_gat_gatv2_seed42_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/run_sprint7_gat_comparison.py \
  --config configs/sweeps/sprint7_gat_gatv2.yaml \
  --run-id "$RUN_ID"
echo "$RUN_ID" > /content/sprint7_gat_gatv2_run_id.txt


In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
ALT_DRIVE_ROOT="/content/drive/MyDrive/crispr-gnn-offtarget"
if [ ! -d "$DRIVE_ROOT" ] && [ -d "$ALT_DRIVE_ROOT" ]; then
  DRIVE_ROOT="$ALT_DRIVE_ROOT"
fi
RUN_BASENAME=$(cat /content/sprint7_gat_gatv2_run_id.txt)
LOCAL_OUT="outputs/sprint7"
RETURN_ROOT="$DRIVE_ROOT/returned_outputs/$RUN_BASENAME"
if [ -e "$RETURN_ROOT" ]; then
  echo "Output already exists in Drive: $RETURN_ROOT" >&2
  exit 1
fi
mkdir -p "$RETURN_ROOT"
rsync -a "$LOCAL_OUT/" "$RETURN_ROOT/"
find "$RETURN_ROOT" -maxdepth 3 -type f | sort | head -80


In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_BASENAME=$(cat /content/sprint7_gat_gatv2_run_id.txt)
OUT="outputs/sprint7"
test -f "$OUT/gat_comparison.csv"
test -f "$OUT/gat_report.md"
test -f "$OUT/gat_run_manifest.json"
test -f "$OUT/graph_artifact_provenance.json"
test -d "$OUT/diagnostics"
test -d "$OUT/figures"
test -f "$OUT/diagnostics/attention_weight_summary.csv"
PYTHONPATH=src uv run python - <<'PY'
import json
from pathlib import Path
manifest = json.loads(Path('outputs/sprint7/gat_run_manifest.json').read_text())
ids = {run['predeclared_id'] for run in manifest['runs']}
expected = {'S7R0_gcn_reference', 'S7R1_gat_edge_aware', 'S7R2_gatv2_edge_aware'}
if ids != expected:
    raise SystemExit(f'Unexpected Sprint 7 run IDs: {ids}')
if manifest.get('optional_runs_executed'):
    raise SystemExit('Unexpected optional run execution in headline notebook')
print('validated', manifest['batch_id'])
PY
